In [ ]:
!pip install -q openai datasets httpx beautifulsoup4

In [ ]:
import asyncio
import hashlib
import json
import os
import random
import re
import tempfile
import time
import warnings
from collections.abc import Awaitable, Callable
from dataclasses import dataclass, field
from pathlib import Path
from urllib.parse import unquote, urlparse

import httpx
from bs4 import BeautifulSoup
from datasets import load_dataset
from openai import AsyncOpenAI

warnings.filterwarnings("ignore")

os.environ["OPENAI_API_KEY"] = ""
os.environ["SERPER_API_KEY"] = ""

client = AsyncOpenAI()
MODEL = "gpt-5-mini"
SEED = 0
NUM_QUESTIONS = 8
MAX_STEPS = 6
READS_PER_STEP = 2
CACHE_DIR = Path(tempfile.gettempdir()) / "decodeai_deep_research"
CACHE_DIR.mkdir(exist_ok=True)
# USD per million tokens, gpt-5-mini list price.
PRICE_IN, PRICE_OUT = 0.25, 2.00

# Deep Research Agent

A deep research agent answers a question from the live web: it searches, reads pages, writes down
evidence with quotes, decides what to search next from what it has learned, and finally writes an
answer whose claims cite that evidence.

---
# 1) Introduction

**The problem.** Take *"If my future wife has the same first name as the 15th First Lady of the
United States' mother and her surname is the same as the second assassinated president's mother's
maiden name, what is my future wife's name?"* Answering needs four facts on different pages, and
the later searches can only be written once the earlier answers are known (you cannot search for
the 15th First Lady's mother until you know who the 15th First Lady was).

| Approach | What goes wrong on multi-hop questions |
|---|---|
| Closed-book model | Guesses the hops it does not remember, with nothing to check against |
| Single search (retrieve, then read) | The question's own words find the first hop at best; later hops need facts not yet known |
| **Research loop** | Each search is written from the notes gathered so far, so it can target the next hop |

**The loop built here.**

```
question
   │
   ▼
┌─> decide: done, or next search query?  ── done ──> write cited answer ──> check citations
│      │ query
│      ▼
│   search ──> read top pages ──> extract notes (claim + verbatim quote + URL)
│      │
└──────┘  notes accumulate in an evidence store outside the prompt
```

| Section | Content |
|---|---|
| 2) Setup | keys, FRAMES questions, model helper with a cost meter |
| 3) The math | why more searches help, and how citations are scored |
| 4) Building the agent | search and fetch tools, evidence notes, the loop, the cited answer, the citation check |
| 5) Walkthrough | the loop on the example question, step by step |
| 6) Small comparison | closed-book vs single search vs research loop on eight questions |
| 7) Practical notes | what production systems add, failure modes |
| 8) Conclusion | summary table |

---
# 2) Setup

Search uses **Serper**, a Google Search API with a free tier. Wikipedia pages are read through the
Wikipedia API, which returns plain text; other pages are fetched over HTTP and reduced to text with
BeautifulSoup. Searches and pages are cached on disk, so a rerun only pays for model calls. Both
keys go in the imports cell at the top. The client is `AsyncOpenAI` so page reads and model calls
can run concurrently.

## 2.1 FRAMES

**FRAMES** (Krishna et al., 2024) has 824 questions written so that the answer needs facts from
2–15 Wikipedia articles. Question 0 is the example above and is used for the walkthrough; eight
random others form the comparison set.

In [ ]:
frames = load_dataset("google/frames-benchmark", split="test")


def frames_item(index: int) -> dict:
    """Return one FRAMES row as id, question and gold answer."""
    row = frames[index]
    return {"id": index, "question": row["Prompt"].strip(), "gold": row["Answer"]}


EXAMPLE = frames_item(0)
QUESTIONS = [
    frames_item(i)
    for i in random.Random(SEED).sample(range(1, len(frames)), NUM_QUESTIONS)
]
print("Q:", EXAMPLE["question"])
print("A:", EXAMPLE["gold"])

## 2.2 Model helper and cost meter

`llm` makes one Chat Completions call; with a JSON schema it uses structured outputs, so every
decision the agent makes comes back as parseable JSON. A `Meter` counts searches, page reads,
tokens and dollars for one run.

In [ ]:
@dataclass
class Meter:
    searches: int = 0
    fetches: int = 0
    input_tokens: int = 0
    output_tokens: int = 0

    @property
    def dollars(self) -> float:
        """Cost of the metered tokens at list price."""
        return (self.input_tokens * PRICE_IN + self.output_tokens * PRICE_OUT) / 1e6


def obj(**properties: dict) -> dict:
    """Strict JSON schema object: every property required, nothing extra."""
    return {
        "type": "object",
        "properties": properties,
        "required": list(properties),
        "additionalProperties": False,
    }


STR, BOOL = {"type": "string"}, {"type": "boolean"}


async def llm(
    meter: Meter, system: str, user: str, schema: dict, effort: str = "low"
) -> dict:
    """One structured-output model call; usage is added to `meter`."""
    response = await client.chat.completions.create(
        model=MODEL,
        reasoning_effort=effort,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {"name": "reply", "strict": True, "schema": schema},
        },
    )
    meter.input_tokens += response.usage.prompt_tokens
    meter.output_tokens += response.usage.completion_tokens
    return json.loads(response.choices[0].message.content)


meter = Meter()
print(await llm(meter, "Answer briefly.", "Capital of Australia?", obj(answer=STR)))
print(meter, f"${meter.dollars:.5f}")

---
# 3) The math

**Why more searches help.** Suppose each search surfaces a page with a needed fact with
probability $p$, and the question needs $h$ facts. Spending $n/h$ searches on each fact, all $h$ are
found with probability

$$
P(\text{all found}) = \left(1 - (1 - p)^{n/h}\right)^{h}.
$$

With $p = 0.4$ and $h = 3$: $n = 3$ searches give $0.4^3 = 0.06$; $n = 6$ give
$(1 - 0.6^2)^3 = 0.26$; $n = 15$ give $(1 - 0.6^5)^3 = 0.78$. Accuracy climbs with the search budget
and then flattens. A single search does worse than this model suggests, because its one query is
the whole question rather than a query aimed at one fact.

**Scoring citations.** The answer is a list of claims $c_1, \dots, c_m$, each citing a set of notes
$E_i$. Citation precision is the fraction of claims that their cited notes actually support:

$$
\text{precision} = \frac{1}{m}\sum_{i=1}^{m} \mathbb{1}\left[E_i \text{ supports } c_i\right].
$$

A model judges "supports" by reading each claim against the quotes of the notes it cites.

---
# 4) Building the agent

## 4.1 Search and fetch

`search` returns title, URL and snippet for the top Google results. Results from sites that host
the benchmark itself (Hugging Face, GitHub, arXiv) are dropped: they contain the questions *with
their answers*, and finding them would measure contamination, not research. `fetch` turns a URL
into plain text.

In [ ]:
BLOCKED_DOMAINS = ("huggingface.co", "github.com", "arxiv.org", "kaggle.com")
MAX_PAGE_CHARS = 20_000
USER_AGENT = "DecodeAI-notebook/1.0 (https://github.com/decodeai; educational demo)"
HTTP = httpx.AsyncClient(
    timeout=20, follow_redirects=True, headers={"User-Agent": USER_AGENT}
)


def cached(kind: str, key: str) -> Path:
    """Cache file for a search query or a page URL."""
    return CACHE_DIR / f"{kind}_{hashlib.sha256(key.encode()).hexdigest()[:24]}.json"


async def search(query: str, meter: Meter) -> list[dict]:
    """Top Google results for `query`, minus benchmark mirrors."""
    meter.searches += 1
    path = cached("search", query)
    if not path.exists():
        response = await HTTP.post(
            "https://google.serper.dev/search",
            headers={"X-API-KEY": os.environ["SERPER_API_KEY"]},
            json={"q": query, "num": 8},
        )
        path.write_text(json.dumps(response.json().get("organic", [])))
    return [
        {"title": r["title"], "url": r["link"], "snippet": r.get("snippet", "")}
        for r in json.loads(path.read_text())
        if not urlparse(r["link"]).netloc.endswith(BLOCKED_DOMAINS)
    ]


meter = Meter()
results = await search("15th First Lady of the United States", meter)
for r in results[:3]:
    print(r["url"], "|", r["snippet"][:70])

Wikipedia refuses scripted page requests without a descriptive User-Agent and asks automated
clients to use its API, so Wikipedia URLs are read through `action=query&prop=extracts`, which
returns the article as plain text. Any other URL is fetched and stripped of scripts, menus and
footers.

In [ ]:
async def fetch_text(url: str) -> str:
    """Plain text of a page; Wikipedia articles via the API."""
    parsed = urlparse(url)
    if parsed.netloc.endswith("wikipedia.org") and parsed.path.startswith("/wiki/"):
        response = await HTTP.get(
            f"https://{parsed.netloc}/w/api.php",
            params={
                "action": "query",
                "prop": "extracts",
                "explaintext": 1,
                "format": "json",
                "redirects": 1,
                "titles": unquote(parsed.path[len("/wiki/") :]),
            },
        )
        pages = response.json()["query"]["pages"]
        return next(iter(pages.values())).get("extract", "")
    response = await HTTP.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
        tag.decompose()
    return (soup.body or soup).get_text(" ")


async def fetch(url: str, meter: Meter) -> str:
    """Cached, whitespace-collapsed page text, truncated to MAX_PAGE_CHARS."""
    meter.fetches += 1
    path = cached("page", url)
    if not path.exists():
        try:
            text = re.sub(r"\s+", " ", await fetch_text(url)).strip()
        except (httpx.HTTPError, ValueError, KeyError):
            text = ""
        path.write_text(json.dumps(text[:MAX_PAGE_CHARS]))
    return json.loads(path.read_text())


page = await fetch(results[0]["url"], meter)
print(len(page), "chars:", page[:300])

## 4.2 Evidence notes with verified quotes

The agent does not keep whole pages in its prompt. For each page it reads, the model extracts a few
**notes**: a one-sentence claim plus a quote copied from the page. Each quote is then checked
against the page text, and a note whose quote is not on the page is thrown away. This single exact
check is what makes the final citations auditable.

In [ ]:
@dataclass
class Note:
    id: int
    claim: str
    quote: str
    url: str


@dataclass
class EvidenceStore:
    notes: list[Note] = field(default_factory=list)
    seen_urls: set[str] = field(default_factory=set)

    def render(self) -> str:
        """Notes as numbered lines for a prompt."""
        return "\n".join(f'[{n.id}] {n.claim} (quote: "{n.quote}")' for n in self.notes)


EXTRACT_SCHEMA = obj(notes={"type": "array", "items": obj(claim=STR, quote=STR)})


def squash(text: str) -> str:
    """Lower-case and collapse whitespace for quote matching."""
    return re.sub(r"\s+", " ", text).strip().lower()


async def read_page(
    question: str, url: str, store: EvidenceStore, meter: Meter
) -> list[Note]:
    """Extract notes from one page; keep only notes whose quote is on the page."""
    page = await fetch(url, meter)
    if len(page) < 200:
        return []
    reply = await llm(
        meter,
        "Extract up to 3 facts from the page that help answer the question. For "
        "each, give a one-sentence claim and a supporting quote copied exactly from "
        "the page (10-40 words). Return an empty list if the page does not help.",
        f"Question: {question}\nPage ({url}):\n{page}",
        EXTRACT_SCHEMA,
        effort="minimal",
    )
    new_notes = []
    for item in reply["notes"]:
        if squash(item["quote"]) in squash(page):
            note = Note(len(store.notes) + 1, item["claim"], item["quote"], url)
            store.notes.append(note)
            new_notes.append(note)
    return new_notes


store, meter = EvidenceStore(), Meter()
new_notes = await read_page(EXAMPLE["question"], results[0]["url"], store, meter)
print(store.render())

## 4.3 The research loop

Each step makes one decision: given the question and the notes so far, either stop (the notes are
enough) or write the next search query. The query is run, the top unread results are read, and
their notes are added. That is the whole agent; production systems run several of these loops in
parallel on sub-questions, but the step is the same.

In [ ]:
NEXT_SCHEMA = obj(done=BOOL, query=STR, reason=STR)


async def research(
    question: str, meter: Meter, max_steps: int = MAX_STEPS, verbose: bool = False
) -> EvidenceStore:
    """Search, read and take notes one step at a time until the model stops."""
    store = EvidenceStore()
    for step in range(1, max_steps + 1):
        decision = await llm(
            meter,
            "You are researching a question one fact at a time. If the notes already "
            "answer it, set done=true. Otherwise write the next web search query "
            "(keywords) for the single most useful missing fact, using names already "
            "found in the notes.",
            f"Question: {question}\nNotes so far:\n{store.render() or '(none)'}",
            NEXT_SCHEMA,
        )
        if verbose:
            print(f"step {step}: {decision['reason']}")
        if decision["done"]:
            break
        results = await search(decision["query"], meter)
        urls = [r["url"] for r in results if r["url"] not in store.seen_urls]
        urls = urls[:READS_PER_STEP]
        store.seen_urls.update(urls)
        note_lists = await asyncio.gather(
            *(read_page(question, url, store, meter) for url in urls)
        )
        if verbose:
            print(f"   search: {decision['query']}")
            for note in (n for notes in note_lists for n in notes):
                print(f"   [{note.id}] {note.claim}")
    return store

## 4.4 Cited answer and citation check

The writer sees the notes, never the pages, and must put note ids after each claim it takes from
them. When the notes miss a fact, it may fill the gap from its own knowledge but must mark that
sentence `[no source]`, so an incomplete search degrades to a closed-book answer instead of a
refusal, and the citation check counts the unsourced claim against precision. The final answer is a
separate short field so grading does not depend on parsing prose.

In [ ]:
ANSWER_SCHEMA = obj(report=STR, final_answer=STR)
CHECK_SCHEMA = obj(claims={"type": "array", "items": obj(claim=STR, supported=BOOL)})


async def write_answer(question: str, store: EvidenceStore, meter: Meter) -> dict:
    """Cited report and short final answer; unsourced claims are marked."""
    return await llm(
        meter,
        "Answer the question from the numbered notes. Write 2-5 sentences with the "
        "note ids in square brackets after every claim, e.g. [2][5]. If the notes miss "
        "a fact, use your own knowledge and end that sentence with [no source]. Then "
        "give the final answer as a short phrase.",
        f"Question: {question}\nNotes:\n{store.render() or '(none)'}",
        ANSWER_SCHEMA,
        effort="medium",
    )


async def citation_precision(report: str, store: EvidenceStore, meter: Meter) -> float:
    """Fraction of the report's claims supported by the notes they cite."""
    reply = await llm(
        meter,
        "Split the report into its factual claims. For each claim, decide whether the "
        "notes it cites (by id in square brackets) support it. A claim with no "
        "citation is unsupported.",
        f"Report:\n{report}\n\nNotes:\n{store.render()}",
        CHECK_SCHEMA,
    )
    claims = reply["claims"]
    return sum(c["supported"] for c in claims) / max(1, len(claims))

---
# 5) Walkthrough on the example question

Each printed step is one decision of the loop: why it continues, what it searches for, and which
notes the pages yielded. Watch the queries change once a name appears in the notes.

In [ ]:
meter = Meter()
start = time.perf_counter()
store = await research(EXAMPLE["question"], meter, verbose=True)
answer = await write_answer(EXAMPLE["question"], store, meter)
seconds = time.perf_counter() - start

print("\nREPORT:", answer["report"])
print("FINAL :", answer["final_answer"], "| GOLD:", EXAMPLE["gold"])
tokens = meter.input_tokens + meter.output_tokens
print(
    f"{meter.searches} searches, {meter.fetches} pages, {len(store.notes)} notes, "
    f"{tokens:,} tokens, ${meter.dollars:.4f}, {seconds:.0f}s"
)
precision = await citation_precision(answer["report"], store, Meter())
print(f"citation precision: {precision:.0%}")

Two things the walkthrough shows. The second query names Garfield only because the first step's
notes established which president and First Lady the question means — the hop-by-hop behaviour a
single search cannot reproduce. And quote verification checks that a quote is *on the page*, not
that the claim reads it correctly: an extractor can attach a real sentence to the wrong person.
That is why the citation check judges each claim against its quotes, and why a conflicting note in
the report is a signal to look again rather than noise.

---
# 6) Small comparison

Three ways to answer the same eight questions:

- **Closed-book**: the model answers from memory.
- **Single search**: search the question once, read the top two pages, answer from their notes.
- **Research loop**: the agent from Section 4.

FRAMES answers are phrased freely ("Jane Ballou", "37 years"), so a model grades each final answer
against the gold answer, the protocol FRAMES itself uses.

In [ ]:
GRADE_SCHEMA = obj(correct=BOOL)


async def grade(question: str, gold: str, predicted: str) -> bool:
    """Model judgement of whether `predicted` means the same as `gold`."""
    reply = await llm(
        Meter(),
        "Reply correct=true if the predicted answer means the same as the gold answer "
        "(ignore phrasing and formatting), else false.",
        f"Question: {question}\nGold: {gold}\nPredicted: {predicted}",
        GRADE_SCHEMA,
        effort="minimal",
    )
    return reply["correct"]


async def closed_book(question: str, meter: Meter) -> str:
    """Answer from the model's memory, no search."""
    reply = await llm(
        meter, "Answer the question.", question, ANSWER_SCHEMA, effort="medium"
    )
    return reply["final_answer"]


async def single_search(question: str, meter: Meter) -> str:
    """Search the question once, read the top pages, answer from their notes."""
    store = EvidenceStore()
    results = await search(question, meter)
    for r in results[:READS_PER_STEP]:
        await read_page(question, r["url"], store, meter)
    return (await write_answer(question, store, meter))["final_answer"]


async def research_loop(question: str, meter: Meter) -> str:
    """Run the research loop, then answer from its notes."""
    store = await research(question, meter)
    return (await write_answer(question, store, meter))["final_answer"]

`evaluate` runs one method on all eight questions concurrently and averages accuracy, searches,
pages, dollars and seconds; the grader's own calls are not counted.

In [ ]:
async def evaluate(solve: Callable[[str, Meter], Awaitable[str]]) -> dict:
    """Run `solve` on every question; return mean metrics and per-question results."""

    async def one(q: dict) -> tuple[bool, Meter, float]:
        """Solve and grade one question."""
        meter, start = Meter(), time.perf_counter()
        predicted = await solve(q["question"], meter)
        seconds = time.perf_counter() - start
        return await grade(q["question"], q["gold"], predicted), meter, seconds

    runs = await asyncio.gather(*(one(q) for q in QUESTIONS))
    num = len(runs)
    return {
        "accuracy": sum(ok for ok, _, _ in runs) / num,
        "searches": sum(m.searches for _, m, _ in runs) / num,
        "pages": sum(m.fetches for _, m, _ in runs) / num,
        "dollars": sum(m.dollars for _, m, _ in runs) / num,
        "seconds": sum(s for _, _, s in runs) / num,
        "correct": [ok for ok, _, _ in runs],
    }


table = {}
for name, solve in [
    ("closed-book", closed_book),
    ("single search", single_search),
    ("research loop", research_loop),
]:
    table[name] = await evaluate(solve)

header = f"{'method':<16}{'acc':>6}{'searches':>10}{'pages':>7}{'$/q':>9}{'sec/q':>7}"
print(header + "  per question")
for name, row in table.items():
    marks = "".join("✓" if ok else "·" for ok in row["correct"])
    print(
        f"{name:<16}{row['accuracy']:>6.0%}{row['searches']:>10.1f}{row['pages']:>7.1f}"
        f"{row['dollars']:>9.4f}{row['seconds']:>7.0f}  {marks}"
    )

Eight questions is a demonstration, not a benchmark: one question is 12.5 points, and reruns move
each row by a question or two. What to look for:

- **Where single search fails.** Its one query is the whole question, so its notes usually cover
  one hop; the rest of the answer comes from memory, and a confidently wrong first hop from a
  page can pull the answer away from what the model would have said closed-book.
- **What the loop costs.** The research loop writes each query from the notes so far, so it can
  reach the later hops, at several searches, about four times the pages and three to four times
  the cost and time per question. The per-question marks show which questions only the loop
  answers.

In [ ]:
await HTTP.aclose()

---
# 7) Practical notes

**What production deep-research agents add to this loop.**

| Addition | Why |
|---|---|
| A planner that splits the question into sub-questions, each with its own loop running in parallel | covers several hops at once; each loop's pages stay out of the others' context |
| A final "is anything missing?" check that can restart research | recovers when the first plan was wrong |
| Hundreds of searches and minutes to an hour per question | hard questions need it; the loop's step is unchanged |
| A model trained with RL on browsing tasks instead of prompted steps | better query writing and stopping decisions |
| Source ranking by reliability, not search position | search ranks popularity |

**Failure modes.**

- *Stopping too early*: the loop accepts a plausible first answer for one hop (e.g. a wrong
  numbering of First Ladies). Asking the stop decision to name a quote for every fact reduces it.
- *Contamination*: benchmark questions appear online with answers; block those sources, as
  `BLOCKED_DOMAINS` does.
- *Prompt injection from pages*: page text is untrusted. Here pages only reach the extractor, which
  returns claims and checked quotes, and the writer never sees raw pages.
- *Blocked fetches*: many sites refuse automated clients (Wikipedia requires a descriptive
  User-Agent and prefers its API). Empty pages silently weaken every method that reads them.

---
# 8) Conclusion

| Step | What it does | Design choice |
|---|---|---|
| Decide | stop, or write the next query from the notes | one structured-output call per step |
| Search | Google results via Serper | cached; benchmark mirrors blocked |
| Read | page → notes with claim + quote | quotes verified against the page text |
| Remember | evidence store outside the prompt | prompt holds notes, not pages |
| Answer | cited report from notes only | separate short final answer for grading |
| Check | citation precision | model judges each claim against its quotes |

**References**

- OpenAI, *Introducing deep research* (2025)
- Anthropic, *How we built our multi-agent research system* (2025)
- Krishna et al., *Fact, Fetch, and Reason: A Unified Evaluation of Retrieval-Augmented Generation (FRAMES)* (2024)
- Wei et al., *BrowseComp: A Simple Yet Challenging Benchmark for Browsing Agents* (2025)
- `dzhng/deep-research`, `assafelovic/gpt-researcher`, `langchain-ai/open_deep_research`